In [1]:
import xesmf as xe
import xarray as xr
import numpy as np

In [2]:
ds = xr.open_dataset(r"W:\VUB\_main_research\data\RMI\RMI_WGS84_2025.nc")

In [4]:
#use xesmf to regrid the data
lat = ds.lat.values
lon = ds.lon.values

ds_out = xr.Dataset({'lat': (['lat'], np.arange(lat.min(), lat.max(), 0.03125), {"units": "degrees_north"}),
                    'lon': (['lon'], np.arange(lon.min(), lon.max(), 0.03125), {"units": "degrees_east"}),
                        }
                    )
# Create regridder
regridder = xe.Regridder(ds, ds_out, 'bilinear')

# Perform the regridding
ds_out = regridder(ds)

In [ ]:
# Define start and end dates
start_date = ds_out.time.values[0]
end_date = ds_out.time.values[-1]

# Generate an array of days
days = np.arange(0, len(ds_out.time))

days = np.arange(start_date, end_date + np.timedelta64(1, 'D'), dtype='datetime64[D]')

# Add the fixed time component '23:00:00' and cast to datetime64[ns]
time_array = (days + np.timedelta64(23, 'h')).astype('datetime64[ns]')

#assign the time to the resampled dataset
ds_out['time'] = time_array

#export each variable to a netcdf file
vars = ['tmax', 'tmin', 'tavg', 'prec', 'pet']

for var in vars:
    ds_out[var].to_netcdf(f"W:/VUB/_main_research/data/RMI/RMI_{var}.nc")